In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
import omicverse as ov
import scanpy as sc
import pandas as pd
import numpy as np
import os

rcParams["figure.figsize"] = (4, 4)


# Dataset1_plot

In [ ]:
adata_rna = sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_1/scanpy_rna.h5ad')
cluster2annotation = {
    'E2Rasgrf2': 'Spatial domain 1',
    'E3Rorb': 'Spatial domain 2',
    'E5Galnt14': 'Spatial domain 4',
    'E4Il1rapl2': 'Spatial domain 3',
}
adata_rna.obs['ground_truth_plot'] = adata_rna.obs['ground_truth'].map(cluster2annotation).astype('category')
adata_rna.uns['ground_truth_plot_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085']
#adata_rna.uns['ground_truth_plot_colors'] = ['#5a5a5a','#ee8227','#eb716b','#79add6']

fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna, 
              color=['ground_truth_plot'], colorbar_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             )  #
ax[0].set_title('Spatial domain', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Main/ground_truth.pdf',dpi=300,bbox_inches='tight')

In [ ]:

adata_rna_pre = sc.read_h5ad('Original_SingleCell_Multiomics_Data/Dataset_1/Chen-2019-RNA.h5ad')
adata_rna_pre.layers['raw'] = adata_rna_pre.X
adata_rna_pre=ov.pp.preprocess(adata_rna_pre,mode='shiftlog|pearson',n_HVGs=3000,
                       target_sum=1e4)
adata_rna_pre.raw = adata_rna_pre
adata_rna_pre = adata_rna_pre[:, adata_rna_pre.var.highly_variable_features]
ov.pp.scale(adata_rna_pre)
ov.pp.pca(adata_rna_pre,layer='scaled',n_pcs=50)
ov.pp.neighbors(adata_rna_pre, n_neighbors=15, n_pcs=50,
               use_rep='scaled|original|X_pca')
ov.pp.umap(adata_rna_pre)

adata_rna_pre.obs_names = [s[:-3] for s in adata_rna_pre.obs_names]
adata_rna.obsm['X_umap'] = adata_rna_pre[adata_rna.obs_names,:].obsm['X_umap']
adata_rna

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.umap(adata_rna,color=['ground_truth_plot'],
                colorbar_loc=None,legend_loc=None,
                ax=ax,show=False)
ax.set_title('Ground truth (Umap)', fontsize=15)
ax.set_xlabel('', fontsize=12)
ax.set_ylabel('', fontsize=12)
fig.savefig('Figure/Main/ground_truth_umap.pdf',dpi=300,bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['ground_truth_plot'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('Ground truth', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Main/ground_truth_without_legend.pdf',dpi=300,bbox_inches='tight')

In [ ]:
# data loading
adata_rna = sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_1/scanpy_rna.h5ad')
cluster2annotation = {
    'E2Rasgrf2': 'Spatial domain 1',
    'E3Rorb': 'Spatial domain 2',
    'E5Galnt14': 'Spatial domain 3',
    'E4Il1rapl2': 'Spatial domain 4',
}
adata_rna.obs['ground_truth_plot'] = adata_rna.obs['ground_truth'].map(cluster2annotation).astype('category')

h5ad_files = [
    'Descart_atac.h5ad','scanpy_rna.h5ad','stagate_rna.h5ad','graphst_rna.h5ad',
    'scglue_multiomics.h5ad','multivi_multiomics.h5ad','spatialglue_multiomics.h5ad','stmultigrn_multiomics.h5ad','cosmos_multiomics.h5ad'
]

embedding_dict = {'Descart_atac.h5ad':'x_pca','scanpy_rna.h5ad':'X_pca','stagate_rna.h5ad':'STAGATE','graphst_rna.h5ad':'GraphST_embedding',
                  'scglue_multiomics.h5ad':'X_glue','multivi_multiomics.h5ad':'X_multivi','spatialglue_multiomics.h5ad':'SpatialGlue',
                 'stmultigrn_multiomics.h5ad':'X_STmultiGAT','cosmos_multiomics.h5ad':'cosmos'}

# Directory containing the h5ad files
data_dir = 'Processed_Simulated_Data/Simulated_Dataset_1/'

# Loop through each h5ad file
for file in h5ad_files:
    adata = sc.read_h5ad(os.path.join(data_dir, file))
    if embedding_dict[file] in adata.obsm:        
        # Copy the specific feature matrix to adata_rna
        if file=='stmultigrn_multiomics.h5ad':
            adata_rna.obsm[file.split('.')[0]] = adata[adata_rna.obs_names,:].obsm[embedding_dict[file]]

        else:
            adata_rna.obsm[file.split('.')[0]] = adata.obsm[embedding_dict[file]]



In [ ]:
ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['scanpy_rna'].shape[1],
                    use_rep='scanpy_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.6)
adata_rna.obs['Scanpy'] = adata_rna.obs['leiden'] 
adata_rna.uns['Scanpy_colors'] = ['#BFD9E5','#E7BD39','#C7A085','#D64F38',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['Scanpy'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('Scanpy', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Main/scanpy.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['stagate_rna'].shape[1],
                    use_rep='stagate_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.3)
adata_rna.obs['STAGATE'] = adata_rna.obs['leiden'] 
adata_rna.uns['STAGATE_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['STAGATE'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('STAGATE', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Main/stagate.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['graphst_rna'].shape[1],
                    use_rep='graphst_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.15)
adata_rna.obs['GraphST'] = adata_rna.obs['leiden'] 
adata_rna.uns['GraphST_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['GraphST'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('GraphST', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Main/graphst.pdf',dpi=300,bbox_inches='tight')  


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['Descart_atac'].shape[1],
                    use_rep='Descart_atac')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.15)
adata_rna.obs['Descart'] = adata_rna.obs['leiden'] 
adata_rna.uns['Descart_colors'] = ['#BFD9E5','#C7A085','#D64F38','#E7BD39',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['Descart'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('Descart', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Main/descart.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['scglue_multiomics'].shape[1],
                    use_rep='scglue_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.6)
adata_rna.obs['scGLUE'] = adata_rna.obs['leiden'] 
adata_rna.uns['scGLUE_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['scGLUE'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('scGLUE', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Main/scglue.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['multivi_multiomics'].shape[1],
                    use_rep='multivi_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.27)
adata_rna.obs['MultiVI'] = adata_rna.obs['leiden'] 
adata_rna.uns['MultiVI_colors'] = ['#BFD9E5','#D64F38','#E7BD39','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['MultiVI'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('MultiVI', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Main/multivi.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['cosmos_multiomics'].shape[1],
                    use_rep='cosmos_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.1)
adata_rna.obs['COSMOS'] = adata_rna.obs['leiden'] 
adata_rna.uns['COSMOS_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['COSMOS'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('COSMOS', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Main/cosmos.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['spatialglue_multiomics'].shape[1],
                    use_rep='spatialglue_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.4)
adata_rna.obs['SpatialGlue'] = adata_rna.obs['leiden'] 
adata_rna.uns['SpatialGlue_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['SpatialGlue'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('SpatialGlue', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Main/SpatialGlue.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['stmultigrn_multiomics'].shape[1],
                    use_rep='stmultigrn_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.6)
adata_rna.obs['STmultiGRN'] = adata_rna.obs['leiden'] 
adata_rna.uns['STmultiGRN_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['STmultiGRN'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('STARNet', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Main/stmultigrn.pdf',dpi=300,bbox_inches='tight')

adata_rna.write_h5ad('Processed_Simulated_Data/Simulated_Dataset_1/adata_all.h5ad')


# Metrics (Data1)

In [ ]:
adata_rna =sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_1/adata_all.h5ad')
adata_rna.obs['STARNet'] = adata_rna.obs['STmultiGRN'].copy()

import numpy as np
from sklearn.metrics import adjusted_rand_score, \
                            adjusted_mutual_info_score, \
                            mutual_info_score,\
                            normalized_mutual_info_score, \
                            homogeneity_score, \
                            v_measure_score


# 真实标签
true_labels = adata_rna.obs['ground_truth_plot']

# 初始化字典来存储每个方法的得分
scores_dict = {
    'Homogeneity': {},
    'MI': {},
    'V_measure': {},
    'AMI': {},
    'NMI': {},
    'ARI': {},
}

# 需要计算得分的方法
methods = ['GraphST', 'COSMOS', 'Descart', 'MultiVI', 'Scanpy', 'scGLUE', 'STAGATE',  'SpatialGlue', 'STARNet']

# 计算每个方法的得分
for method in methods:
    predicted_labels = adata_rna.obs[method]
    
    scores_dict['ARI'][method] = adjusted_rand_score(true_labels, predicted_labels)
    scores_dict['AMI'][method] = adjusted_mutual_info_score(true_labels, predicted_labels)
    scores_dict['NMI'][method] = normalized_mutual_info_score(true_labels, predicted_labels)
    scores_dict['Homogeneity'][method] = homogeneity_score(true_labels, predicted_labels)
    scores_dict['MI'][method] = mutual_info_score(true_labels, predicted_labels)
    scores_dict['V_measure'][method] = v_measure_score(true_labels, predicted_labels)

# 输出字典
print(scores_dict)

In [ ]:
scores_dict['V_measure']

In [ ]:
scores_dict['AMI']

In [ ]:
scores_dict['NMI']

In [ ]:
#https://www.xiaohongshu.com/explore/63e0bbea000000001a026ccb?app_platform=android&ignoreEngage=true&app_version=8.68.5&share_from_user_hidden=true&xsec_source=app_share&type=normal&xsec_token=CBCu2gJ8hZImMrDNpXItSLAXyfCEuEvY1TCQS24_A0UoY=&
#author_share=1&xhsshare=WeixinSession&shareRedId=ODhFOTY2OEI2NzUyOTgwNjZIOTdKO0s5&apptime=1736420335&share_id=cb4ebd2757e948378fb341a8c0f14e66

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

scores_df = pd.DataFrame(scores_dict)

scores_long_df = pd.melt(scores_df.reset_index(), id_vars='index', var_name='method', value_name='score')
scores_long_df.rename(columns={'index': 'model'}, inplace=True)

g = sns.catplot(
    data=scores_long_df, kind="bar",legend=False,
    x="method", y="score", hue="model",
    errorbar="sd", alpha=0.8, height=6,
    palette=['#78B41B', '#1B78B4', '#F67F20', '#9368AD', '#8C574B', '#F6B475', '#6EC2A2', '#939597', '#B83945',]
)

# 设置边框与刻度线
g.ax.spines['left'].set_linewidth(1.5)
g.ax.spines['bottom'].set_linewidth(1.5)
g.ax.tick_params(axis='x', labelsize=15, width=1.5,length=4) 
g.ax.tick_params(axis='y', labelsize=15, width=1.5,length=4) 

# x轴坐标和指标名称
g.set_axis_labels("", "value", fontsize=15,)


# 创建带有圆形标记的线条，用于自定义图例
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker='s', color='white', label=model,alpha=0.8,
                  markerfacecolor=color, markersize=12)
           for model, color in zip(scores_long_df['model'].unique(), ['#78B41B', '#1B78B4', '#F67F20', '#9368AD', '#8C574B', '#F6B475', '#6EC2A2', '#939597', '#B83945',])]
# 设置自定义图例
g.fig.legend(handles=handles, bbox_to_anchor=(1.22, 0.85), title_fontsize=15, fontsize=15)

g.fig.set_size_inches(7.5, 4)

# 字体大小
plt.ylim(0,1.0)
g.set_xticklabels(fontsize=15, rotation=0)
g.set_yticklabels(fontsize=15, )
plt.grid(False)
plt.tight_layout()
#plt.title('Model Scores by Method', fontsize=15, fontweight='bold')

plt.show()
g.savefig('Figure/Main/data1_benchmarks.pdf',dpi=300,bbox_inches='tight')

# Supplementary

## Dataset2

In [ ]:
adata_rna = sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_2/scanpy_rna.h5ad')
cluster2annotation = {
    'HSC': 'Spatial domain 1',
    'HMP': 'Spatial domain 2',
    'Mono': 'Spatial domain 3',
    'CLP': 'Spatial domain 4',
}
adata_rna.obs['ground_truth_plot'] = adata_rna.obs['ground_truth'].map(cluster2annotation).astype('category')
new_order = ['Spatial domain 1','Spatial domain 2','Spatial domain 3','Spatial domain 4',]
adata_rna.obs['ground_truth_plot'] = adata_rna.obs['ground_truth_plot'].cat.reorder_categories(new_order)
adata_rna.uns['ground_truth_plot_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085']
#adata_rna.uns['ground_truth_plot_colors'] = ['#5a5a5a','#ee8227','#eb716b','#79add6']

fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna, 
              color=['ground_truth_plot'], colorbar_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             )  #
ax[0].set_title('Spatial domain', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_2/ground_truth.pdf',dpi=300,bbox_inches='tight')

In [ ]:
adata_rna = sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_2/scanpy_rna.h5ad')
cluster2annotation = {
    'HSC': 'Spatial domain 1',
    'HMP': 'Spatial domain 2',
    'Mono': 'Spatial domain 3',
    'CLP': 'Spatial domain 4',
}
adata_rna.obs['ground_truth_plot'] = adata_rna.obs['ground_truth'].map(cluster2annotation).astype('category')
new_order = ['Spatial domain 1','Spatial domain 2','Spatial domain 3','Spatial domain 4',]
adata_rna.obs['ground_truth_plot'] = adata_rna.obs['ground_truth_plot'].cat.reorder_categories(new_order)
adata_rna.uns['ground_truth_plot_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085']

h5ad_files = [
    'Descart_atac.h5ad','scanpy_rna.h5ad','stagate_rna.h5ad','graphst_rna.h5ad',
    'scglue_multiomics.h5ad','multivi_multiomics.h5ad','spatialglue_multiomics.h5ad','stmultigrn_multiomics.h5ad','cosmos_multiomics.h5ad'
]

embedding_dict = {'Descart_atac.h5ad':'x_pca','scanpy_rna.h5ad':'X_pca','stagate_rna.h5ad':'STAGATE','graphst_rna.h5ad':'GraphST_embedding',
                  'scglue_multiomics.h5ad':'X_glue','multivi_multiomics.h5ad':'X_multivi','spatialglue_multiomics.h5ad':'SpatialGlue',
                 'stmultigrn_multiomics.h5ad':'X_STmultiGAT','cosmos_multiomics.h5ad':'cosmos'}

# Directory containing the h5ad files
data_dir = 'Processed_Simulated_Data/Simulated_Dataset_2/'

# Loop through each h5ad file
for file in h5ad_files:
    adata = sc.read_h5ad(os.path.join(data_dir, file))
    if embedding_dict[file] in adata.obsm:        
        # Copy the specific feature matrix to adata_rna
        if file=='stmultigrn_multiomics.h5ad':
            adata_rna.obsm[file.split('.')[0]] = adata[adata_rna.obs_names,:].obsm[embedding_dict[file]]

        else:
            adata_rna.obsm[file.split('.')[0]] = adata.obsm[embedding_dict[file]]

adata_rna

In [ ]:
ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['scanpy_rna'].shape[1],
                    use_rep='scanpy_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.5)
adata_rna.obs['Scanpy'] = adata_rna.obs['leiden'] 
adata_rna.uns['Scanpy_colors'] = ['#BFD9E5', '#D64F38','#E7BD39','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['Scanpy'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('Scanpy', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_2/scanpy.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['stagate_rna'].shape[1],
                    use_rep='stagate_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.3)
adata_rna.obs['STAGATE'] = adata_rna.obs['leiden'] 
adata_rna.uns['STAGATE_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['STAGATE'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('STAGATE', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_2/stagate.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['graphst_rna'].shape[1],
                    use_rep='graphst_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.2)
adata_rna.obs['GraphST'] = adata_rna.obs['leiden'] 
adata_rna.uns['GraphST_colors'] = ['#E7BD39', '#BFD9E5', '#C7A085','#D64F38',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['GraphST'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('GraphST', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_2/graphst.pdf',dpi=300,bbox_inches='tight')  



ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['Descart_atac'].shape[1],
                    use_rep='Descart_atac')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.12)
adata_rna.obs['Descart'] = adata_rna.obs['leiden'] 
adata_rna.uns['Descart_colors'] = ['#BFD9E5','#E7BD39', '#D64F38', '#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['Descart'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('Descart', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_2/descart.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['scglue_multiomics'].shape[1],
                    use_rep='scglue_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.4)
adata_rna.obs['scGLUE'] = adata_rna.obs['leiden'] 
adata_rna.uns['scGLUE_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['scGLUE'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('scGLUE', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_2/scglue.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['multivi_multiomics'].shape[1],
                    use_rep='multivi_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.35)
adata_rna.obs['MultiVI'] = adata_rna.obs['leiden'] 
adata_rna.uns['MultiVI_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['MultiVI'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('MultiVI', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_2/multivi.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['cosmos_multiomics'].shape[1],
                    use_rep='cosmos_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.15)
adata_rna.obs['COSMOS'] = adata_rna.obs['leiden'] 
adata_rna.uns['COSMOS_colors'] = ['#BFD9E5','#D64F38','#C7A085','#E7BD39',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['COSMOS'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('COSMOS', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_2/cosmos.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['spatialglue_multiomics'].shape[1],
                    use_rep='spatialglue_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.25)
adata_rna.obs['SpatialGlue'] = adata_rna.obs['leiden'] 
adata_rna.uns['SpatialGlue_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['SpatialGlue'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('SpatialGlue', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_2/SpatialGlue.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['stmultigrn_multiomics'].shape[1],
                    use_rep='stmultigrn_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.6)
adata_rna.obs['STmultiGRN'] = adata_rna.obs['leiden'] 
adata_rna.uns['STmultiGRN_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['STmultiGRN'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('STARNet', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_2/stmultigrn.pdf',dpi=300,bbox_inches='tight')

adata_rna.write_h5ad('Processed_Simulated_Data/Simulated_Dataset_2/adata_all.h5ad')


## Dataset3

In [ ]:
adata_rna = sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_3/scanpy_rna.h5ad')
sc.pl.spatial(adata_rna, 
              color=['ground_truth'], colorbar_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
           #   ax=ax,show=False
             )  #

In [ ]:
adata_rna = sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_3/scanpy_rna.h5ad')
cluster2annotation = {
    'E5Parm1': 'Spatial domain 1',
    'E5Sulf1': 'Spatial domain 2',
    'E6Tle4': 'Spatial domain 3',
    'OliM': 'Spatial domain 4',
}
adata_rna.obs['ground_truth_plot'] = adata_rna.obs['ground_truth'].map(cluster2annotation).astype('category')
adata_rna.uns['ground_truth_plot_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085']

fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna, 
              color=['ground_truth_plot'], colorbar_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             )  #
ax[0].set_title('Spatial domain', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_3/ground_truth.pdf',dpi=300,bbox_inches='tight')

In [ ]:
adata_rna = sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_3/scanpy_rna.h5ad')
cluster2annotation = {
    'E5Parm1': 'Spatial domain 1',
    'E5Sulf1': 'Spatial domain 2',
    'E6Tle4': 'Spatial domain 3',
    'OliM': 'Spatial domain 4',
}
adata_rna.obs['ground_truth_plot'] = adata_rna.obs['ground_truth'].map(cluster2annotation).astype('category')
#new_order = ['Spatial domain 1','Spatial domain 2','Spatial domain 3','Spatial domain 4',]
#adata_rna.obs['ground_truth_plot'] = adata_rna.obs['ground_truth_plot'].cat.reorder_categories(new_order)
adata_rna.uns['ground_truth_plot_colors'] = ['#BFD9E5','#E7BD39','#D64F38','#C7A085']

h5ad_files = [
    'Descart_atac.h5ad','scanpy_rna.h5ad','stagate_rna.h5ad','graphst_rna.h5ad',
    'scglue_multiomics.h5ad','multivi_multiomics.h5ad','spatialglue_multiomics.h5ad','stmultigrn_multiomics.h5ad','cosmos_multiomics.h5ad'
]

embedding_dict = {'Descart_atac.h5ad':'x_pca','scanpy_rna.h5ad':'X_pca','stagate_rna.h5ad':'STAGATE','graphst_rna.h5ad':'GraphST_embedding',
                  'scglue_multiomics.h5ad':'X_glue','multivi_multiomics.h5ad':'X_multivi','spatialglue_multiomics.h5ad':'SpatialGlue',
                 'stmultigrn_multiomics.h5ad':'X_STmultiGAT','cosmos_multiomics.h5ad':'cosmos'}

# Directory containing the h5ad files
data_dir = 'Processed_Simulated_Data/Simulated_Dataset_3/'

# Loop through each h5ad file
for file in h5ad_files:
    adata = sc.read_h5ad(os.path.join(data_dir, file))
    if embedding_dict[file] in adata.obsm:        
        # Copy the specific feature matrix to adata_rna
        if file=='stmultigrn_multiomics.h5ad':
            adata_rna.obsm[file.split('.')[0]] = adata[adata_rna.obs_names,:].obsm[embedding_dict[file]]

        else:
            adata_rna.obsm[file.split('.')[0]] = adata.obsm[embedding_dict[file]]

adata_rna

In [ ]:
ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['scanpy_rna'].shape[1],
                    use_rep='scanpy_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.5)
adata_rna.obs['Scanpy'] = adata_rna.obs['leiden'] 
adata_rna.uns['Scanpy_colors'] = ['#D64F38','#E7BD39','#BFD9E5','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['Scanpy'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('Scanpy', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_3/scanpy.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['stagate_rna'].shape[1],
                    use_rep='stagate_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.4)
adata_rna.obs['STAGATE'] = adata_rna.obs['leiden'] 
adata_rna.uns['STAGATE_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['STAGATE'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('STAGATE', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_3/stagate.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['graphst_rna'].shape[1],
                    use_rep='graphst_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.11)
adata_rna.obs['GraphST'] = adata_rna.obs['leiden'] 
adata_rna.uns['GraphST_colors'] = ['#D64F38','#E7BD39','#BFD9E5','#C7A085']
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['GraphST'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('GraphST', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_3/graphst.pdf',dpi=300,bbox_inches='tight')  



ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['Descart_atac'].shape[1],
                    use_rep='Descart_atac')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.08)
adata_rna.obs['Descart'] = adata_rna.obs['leiden'] 
adata_rna.uns['Descart_colors'] = ['#D64F38','#E7BD39','#BFD9E5','#C7A085']
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['Descart'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('Descart', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_3/descart.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['scglue_multiomics'].shape[1],
                    use_rep='scglue_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.4)
adata_rna.obs['scGLUE'] = adata_rna.obs['leiden'] 
adata_rna.uns['scGLUE_colors'] = ['#D64F38','#BFD9E5','#C7A085','#E7BD39',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['scGLUE'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('scGLUE', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_3/scglue.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['multivi_multiomics'].shape[1],
                    use_rep='multivi_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.3)
adata_rna.obs['MultiVI'] = adata_rna.obs['leiden'] 
adata_rna.uns['MultiVI_colors'] =  ['#D64F38','#BFD9E5','#E7BD39','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['MultiVI'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('MultiVI', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_3/multivi.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['cosmos_multiomics'].shape[1],
                    use_rep='cosmos_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.05)
adata_rna.obs['COSMOS'] = adata_rna.obs['leiden'] 
adata_rna.uns['COSMOS_colors'] = ['#D64F38','#C7A085','#BFD9E5','#E7BD39',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['COSMOS'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('COSMOS', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_3/cosmos.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['spatialglue_multiomics'].shape[1],
                    use_rep='spatialglue_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.25)
adata_rna.obs['SpatialGlue'] = adata_rna.obs['leiden'] 
adata_rna.uns['SpatialGlue_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085']
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['SpatialGlue'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('SpatialGlue', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_3/SpatialGlue.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['stmultigrn_multiomics'].shape[1],
                    use_rep='stmultigrn_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.6)
adata_rna.obs['STmultiGRN'] = adata_rna.obs['leiden'] 
adata_rna.uns['STmultiGRN_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085']
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['STmultiGRN'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('STARNet', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_3/stmultigrn.pdf',dpi=300,bbox_inches='tight')

adata_rna.write_h5ad('Processed_Simulated_Data/Simulated_Dataset_3/adata_all.h5ad')


## Dataset4

In [ ]:
adata_rna = sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_4/scanpy_rna.h5ad')
sc.pl.spatial(adata_rna, 
              color=['ground_truth'], colorbar_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
           #   ax=ax,show=False
             )  #

In [ ]:
adata_rna = sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_4/scanpy_rna.h5ad')
cluster2annotation = {
    'CLP': 'Spatial domain 1',
    'HMP': 'Spatial domain 2',
    'HSC': 'Spatial domain 3',
    'Mono': 'Spatial domain 4',
}
adata_rna.obs['ground_truth_plot'] = adata_rna.obs['ground_truth'].map(cluster2annotation).astype('category')
adata_rna.uns['ground_truth_plot_colors'] = ['#E7BD39','#BFD9E5','#D64F38','#C7A085']

fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna, 
              color=['ground_truth_plot'], colorbar_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             )  #
ax[0].set_title('Spatial domain', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_4/ground_truth.pdf',dpi=300,bbox_inches='tight')

In [ ]:
adata_rna = sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_4/scanpy_rna.h5ad')
cluster2annotation = {
    'CLP': 'Spatial domain 1',
    'HMP': 'Spatial domain 2',
    'HSC': 'Spatial domain 3',
    'Mono': 'Spatial domain 4',
}
adata_rna.obs['ground_truth_plot'] = adata_rna.obs['ground_truth'].map(cluster2annotation).astype('category')
adata_rna.uns['ground_truth_plot_colors'] = ['#E7BD39','#BFD9E5','#D64F38','#C7A085']

h5ad_files = [
    'Descart_atac.h5ad','scanpy_rna.h5ad','stagate_rna.h5ad','graphst_rna.h5ad',
    'scglue_multiomics.h5ad','multivi_multiomics.h5ad','spatialglue_multiomics.h5ad','stmultigrn_multiomics.h5ad','cosmos_multiomics.h5ad'
]

embedding_dict = {'Descart_atac.h5ad':'x_pca','scanpy_rna.h5ad':'X_pca','stagate_rna.h5ad':'STAGATE','graphst_rna.h5ad':'GraphST_embedding',
                  'scglue_multiomics.h5ad':'X_glue','multivi_multiomics.h5ad':'X_multivi','spatialglue_multiomics.h5ad':'SpatialGlue',
                 'stmultigrn_multiomics.h5ad':'X_STmultiGAT','cosmos_multiomics.h5ad':'cosmos'}

# Directory containing the h5ad files
data_dir = 'Processed_Simulated_Data/Simulated_Dataset_4/'

# Loop through each h5ad file
for file in h5ad_files:
    adata = sc.read_h5ad(os.path.join(data_dir, file))
    if embedding_dict[file] in adata.obsm:        
        # Copy the specific feature matrix to adata_rna
        if file=='stmultigrn_multiomics.h5ad':
            adata_rna.obsm[file.split('.')[0]] = adata[adata_rna.obs_names,:].obsm[embedding_dict[file]]

        else:
            adata_rna.obsm[file.split('.')[0]] = adata.obsm[embedding_dict[file]]

adata_rna

In [ ]:
ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['scanpy_rna'].shape[1],
                    use_rep='scanpy_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.4)
adata_rna.obs['Scanpy'] = adata_rna.obs['leiden'] 
adata_rna.uns['Scanpy_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['Scanpy'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('Scanpy', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_4/scanpy.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['stagate_rna'].shape[1],
                    use_rep='stagate_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.4)
adata_rna.obs['STAGATE'] = adata_rna.obs['leiden'] 
adata_rna.uns['STAGATE_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['STAGATE'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('STAGATE', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_4/stagate.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['graphst_rna'].shape[1],
                    use_rep='graphst_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.18)
adata_rna.obs['GraphST'] = adata_rna.obs['leiden'] 
adata_rna.uns['GraphST_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['GraphST'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('GraphST', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_4/graphst.pdf',dpi=300,bbox_inches='tight')  



ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['Descart_atac'].shape[1],
                    use_rep='Descart_atac')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.062)
adata_rna.obs['Descart'] = adata_rna.obs['leiden'] 
adata_rna.uns['Descart_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085']
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['Descart'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('Descart', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_4/descart.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['scglue_multiomics'].shape[1],
                    use_rep='scglue_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.4)
adata_rna.obs['scGLUE'] = adata_rna.obs['leiden'] 
adata_rna.uns['scGLUE_colors'] = ['#D64F38','#BFD9E5','#C7A085','#E7BD39',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['scGLUE'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('scGLUE', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_4/scglue.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['multivi_multiomics'].shape[1],
                    use_rep='multivi_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.3)
adata_rna.obs['MultiVI'] = adata_rna.obs['leiden'] 
adata_rna.uns['MultiVI_colors'] =  ['#D64F38','#BFD9E5','#C7A085','#E7BD39',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['MultiVI'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('MultiVI', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_4/multivi.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['cosmos_multiomics'].shape[1],
                    use_rep='cosmos_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.06)
adata_rna.obs['COSMOS'] = adata_rna.obs['leiden'] 
adata_rna.uns['COSMOS_colors'] = ['#BFD9E5','#D64F38','#E7BD39','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['COSMOS'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('COSMOS', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_4/cosmos.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['spatialglue_multiomics'].shape[1],
                    use_rep='spatialglue_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.25)
adata_rna.obs['SpatialGlue'] = adata_rna.obs['leiden'] 
adata_rna.uns['SpatialGlue_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085']
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['SpatialGlue'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('SpatialGlue', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_4/SpatialGlue.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['stmultigrn_multiomics'].shape[1],
                    use_rep='stmultigrn_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.6)
adata_rna.obs['STmultiGRN'] = adata_rna.obs['leiden'] 
adata_rna.uns['STmultiGRN_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085']
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['STmultiGRN'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('STARNet', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_4/stmultigrn.pdf',dpi=300,bbox_inches='tight')

adata_rna.write_h5ad('Processed_Simulated_Data/Simulated_Dataset_4/adata_all.h5ad')
del adata_rna

## Dataset_5

In [ ]:
adata_rna = sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_5/scanpy_rna.h5ad')
sc.pl.spatial(adata_rna, 
              color=['ground_truth'], colorbar_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
           #   ax=ax,show=False
             )  #

In [ ]:
adata_rna = sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_5/scanpy_rna.h5ad')
cluster2annotation = {
    'Ery': 'Spatial domain 1',
    'HMP': 'Spatial domain 2',
    'HSC': 'Spatial domain 3',
    'MEP': 'Spatial domain 4',
}
adata_rna.obs['ground_truth_plot'] = adata_rna.obs['ground_truth'].map(cluster2annotation).astype('category')
adata_rna.uns['ground_truth_plot_colors'] = ['#C7A085','#BFD9E5','#D64F38','#E7BD39',]

fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna, 
              color=['ground_truth_plot'], colorbar_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             )  #
ax[0].set_title('Spatial domain', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_5/ground_truth.pdf',dpi=300,bbox_inches='tight')

In [ ]:
adata_rna = sc.read_h5ad('Processed_Simulated_Data/Simulated_Dataset_5/scanpy_rna.h5ad')
cluster2annotation = {
    'Ery': 'Spatial domain 1',
    'HMP': 'Spatial domain 2',
    'HSC': 'Spatial domain 3',
    'MEP': 'Spatial domain 4',
}
adata_rna.obs['ground_truth_plot'] = adata_rna.obs['ground_truth'].map(cluster2annotation).astype('category')
adata_rna.uns['ground_truth_plot_colors'] = ['#E7BD39','#BFD9E5','#D64F38','#C7A085']

h5ad_files = [
    'Descart_atac.h5ad','scanpy_rna.h5ad','stagate_rna.h5ad','graphst_rna.h5ad',
    'scglue_multiomics.h5ad','multivi_multiomics.h5ad','spatialglue_multiomics.h5ad','stmultigrn_multiomics.h5ad','cosmos_multiomics.h5ad'
]

embedding_dict = {'Descart_atac.h5ad':'x_pca','scanpy_rna.h5ad':'X_pca','stagate_rna.h5ad':'STAGATE','graphst_rna.h5ad':'GraphST_embedding',
                  'scglue_multiomics.h5ad':'X_glue','multivi_multiomics.h5ad':'X_multivi','spatialglue_multiomics.h5ad':'SpatialGlue',
                 'stmultigrn_multiomics.h5ad':'X_STmultiGAT','cosmos_multiomics.h5ad':'cosmos'}

# Directory containing the h5ad files
data_dir = 'Processed_Simulated_Data/Simulated_Dataset_5/'

# Loop through each h5ad file
for file in h5ad_files:
    adata = sc.read_h5ad(os.path.join(data_dir, file))
    if embedding_dict[file] in adata.obsm:        
        # Copy the specific feature matrix to adata_rna
        if file=='stmultigrn_multiomics.h5ad':
            adata_rna.obsm[file.split('.')[0]] = adata[adata_rna.obs_names,:].obsm[embedding_dict[file]]

        else:
            adata_rna.obsm[file.split('.')[0]] = adata.obsm[embedding_dict[file]]

adata_rna

In [ ]:
ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['scanpy_rna'].shape[1],
                    use_rep='scanpy_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.4)
adata_rna.obs['Scanpy'] = adata_rna.obs['leiden'] 
adata_rna.uns['Scanpy_colors'] = ['#D64F38','#E7BD39','#BFD9E5','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['Scanpy'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('Scanpy', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_5/scanpy.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['stagate_rna'].shape[1],
                    use_rep='stagate_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.4)
adata_rna.obs['STAGATE'] = adata_rna.obs['leiden'] 
adata_rna.uns['STAGATE_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['STAGATE'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('STAGATE', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_5/stagate.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['graphst_rna'].shape[1],
                    use_rep='graphst_rna')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.18)
adata_rna.obs['GraphST'] = adata_rna.obs['leiden'] 
adata_rna.uns['GraphST_colors'] = ['#D64F38','#E7BD39','#C7A085','#BFD9E5',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['GraphST'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('GraphST', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_5/graphst.pdf',dpi=300,bbox_inches='tight')  



ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['Descart_atac'].shape[1],
                    use_rep='Descart_atac')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.05)
adata_rna.obs['Descart'] = adata_rna.obs['leiden'] 
adata_rna.uns['Descart_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085']
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['Descart'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('Descart', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_5/descart.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['scglue_multiomics'].shape[1],
                    use_rep='scglue_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.4)
adata_rna.obs['scGLUE'] = adata_rna.obs['leiden'] 
adata_rna.uns['scGLUE_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['scGLUE'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('scGLUE', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_5/scglue.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['multivi_multiomics'].shape[1],
                    use_rep='multivi_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.3)
adata_rna.obs['MultiVI'] = adata_rna.obs['leiden'] 
adata_rna.uns['MultiVI_colors'] =  ['#D64F38','#BFD9E5','#E7BD39','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['MultiVI'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('MultiVI', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_5/multivi.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['cosmos_multiomics'].shape[1],
                    use_rep='cosmos_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.05)
adata_rna.obs['COSMOS'] = adata_rna.obs['leiden'] 
adata_rna.uns['COSMOS_colors'] = ['#D64F38','#E7BD39','#BFD9E5','#C7A085',]
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['COSMOS'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('COSMOS', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_5/cosmos.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['spatialglue_multiomics'].shape[1],
                    use_rep='spatialglue_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.25)
adata_rna.obs['SpatialGlue'] = adata_rna.obs['leiden'] 
adata_rna.uns['SpatialGlue_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085']
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['SpatialGlue'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('SpatialGlue', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_5/SpatialGlue.pdf',dpi=300,bbox_inches='tight')


ov.pp.neighbors(adata_rna, n_neighbors=15, n_pcs=adata_rna.obsm['stmultigrn_multiomics'].shape[1],
                    use_rep='stmultigrn_multiomics')
ov.utils.cluster(adata_rna, method='leiden', resolution=0.6)
adata_rna.obs['STmultiGRN'] = adata_rna.obs['leiden'] 
adata_rna.uns['STmultiGRN_colors'] = ['#D64F38','#BFD9E5','#E7BD39','#C7A085']
fig, ax = plt.subplots(figsize=(3, 3))
ax = sc.pl.spatial(adata_rna,
              color=['STmultiGRN'], colorbar_loc=None,legend_loc=None,
              ncols=3, spot_size=0.125,legend_fontsize=15,
              ax=ax,show=False
             ) 
ax[0].set_title('STARNet', fontsize=15)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
ax[0].set_xlim(xmin + 0.04*(xmax-xmin), xmax - 0.04*(xmax-xmin))  # 扩展 10%
ax[0].set_ylim(ymin + 0.04*(ymax-ymin), ymax - 0.04*(ymax-ymin))
ax[0].set_xlabel('', fontsize=12)
ax[0].set_ylabel('', fontsize=12)
fig.savefig('Figure/Supplementary/Dataset_5/stmultigrn.pdf',dpi=300,bbox_inches='tight')

adata_rna.write_h5ad('Processed_Simulated_Data/Simulated_Dataset_5/adata_all.h5ad')


# Metrics (5 datasets)

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score, \
                            adjusted_mutual_info_score, \
                            mutual_info_score,\
                            normalized_mutual_info_score, \
                            homogeneity_score, \
                            v_measure_score

# 定义文件路径和方法列表
file_paths = [
    'Processed_Simulated_Data/Simulated_Dataset_1/adata_all.h5ad',
    'Processed_Simulated_Data/Simulated_Dataset_2/adata_all.h5ad',
    'Processed_Simulated_Data/Simulated_Dataset_3/adata_all.h5ad',
    'Processed_Simulated_Data/Simulated_Dataset_4/adata_all.h5ad',
    'Processed_Simulated_Data/Simulated_Dataset_5/adata_all.h5ad'
]
methods = ['GraphST', 'COSMOS', 'Descart', 'MultiVI', 'Scanpy', 'scGLUE', 'STAGATE', 'SpatialGlue', 'STARNet']

# 初始化得分列表
scores_list = []

# 遍历每个文件
for file_path in file_paths:
    # 读取数据
    adata_rna = sc.read_h5ad(file_path)
    
    # 获取真实标签
    true_labels = adata_rna.obs['ground_truth_plot']

    adata_rna.obs['STARNet'] = adata_rna.obs['STmultiGRN']
    # 计算每个方法的得分
    for method in methods:
        predicted_labels = adata_rna.obs[method]

        scores = {
            'file': file_path,
            'method': method,
            'Homogeneity': homogeneity_score(true_labels, predicted_labels),
            'MI': mutual_info_score(true_labels, predicted_labels),
            'V_measure': v_measure_score(true_labels, predicted_labels),
            'AMI': adjusted_mutual_info_score(true_labels, predicted_labels),
            'NMI': normalized_mutual_info_score(true_labels, predicted_labels),
            'ARI': adjusted_rand_score(true_labels, predicted_labels),
        }
        
        scores_list.append(scores)

# 将得分列表转换为DataFrame
scores_df = pd.DataFrame(scores_list)

# 将DataFrame转换为长格式，以便绘制箱线图
scores_long_df = pd.melt(scores_df, id_vars=['file', 'method'], var_name='metric', value_name='value')


In [ ]:
# 绘制箱线图
plt.figure(figsize=(7.5, 4))

g = sns.boxplot(data=scores_long_df, x='metric', y='value', hue='method',width=0.8,boxprops=dict(alpha=0.8),legend=False,
                palette=['#78B41B', '#1B78B4', '#F67F20', '#9368AD', '#8C574B', '#F6B475', '#6EC2A2', '#939597', '#B83945',],
                showfliers=False)

# 设置边框与刻度线
g.spines['left'].set_linewidth(1.5)
g.spines['bottom'].set_linewidth(1.5)
g.spines['right'].set_linewidth(0)
g.spines['top'].set_linewidth(0)

g.tick_params(axis='x', labelsize=15, width=1.5,length=4) 
g.tick_params(axis='y', labelsize=15, width=1.5,length=4) 
plt.ylim(0, 1.3)
g.set_xticklabels(g.get_xticklabels(), fontsize=15, )
g.set_yticklabels(g.get_yticklabels(), fontsize=15, )
g.set_ylabel("Value", fontsize=15,)
g.set_xlabel("", fontsize=15,)

plt.grid(False)
plt.tight_layout()

plt.savefig('Figure/Main/5datasets_benchmarks.pdf',dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
!pip list